In [1]:
%%capture
# Installs Unsloth, Xformers (Flash Attention) and all other packages!
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# We have to check which Torch version for Xformers (2.3 -> 0.0.27)
from torch import __version__; from packaging.version import Version as V
xformers = "xformers==0.0.27" if V(__version__) < V("2.4.0") else "xformers"
!pip install --no-deps {xformers} trl peft accelerate bitsandbytes triton
!pip install Pillow

!pip install sentence_transformers

In [2]:
import requests
import json
import csv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.font_manager as fm
import matplotlib.image as mpimg
from io import StringIO, BytesIO
import PIL


# url = 'manual_add.json'
# response = requests.get(url)


# logo_banner = 'horizontal-logo.png'
# logo_response = requests.get(logo_banner)
logo_img = mpimg.imread('horizontal-logo.png')



import json
with open('final_file.json', 'r') as json_file:
    data = json.load(json_file)

# Load the CSV string into a pandas DataFrame
df = pd.read_csv('Joined_DF_0.csv')
df['action_time'] = pd.to_datetime(df['action_time'],format='ISO8601')
df['Date']=pd.to_datetime(df['Date'])
manjari_bold_path = 'Manjari-Bold.ttf'
manjari_regular_path = 'Manjari-Regular.ttf'
manjari_thin_path = 'Manjari-thin.ttf'

dark = "#3B3365"
medium = "#8F81DD"
light = "#BCADFF"
peach = "#FFEEDD"

# Load the custom font
manjari_bold = fm.FontProperties(fname=manjari_bold_path)
manjari_regular = fm.FontProperties(fname=manjari_regular_path)
manjari_thin = fm.FontProperties(fname=manjari_thin_path)
df

,staff,replaced,Date,action_time,product,type,quantity,base,total
0,Amanda,NaN,2024-06-17,2024-06-17 10:10:30,Blue Machine,Photo,1.0,10.0,10.0
1,Amanda,NaN,2024-06-17,2024-06-17 10:13:38,Blue Machine,Photo,1.0,10.0,10.0
2,Amanda,NaN,2024-06-17,2024-06-17 10:24:14,Purple Machine,Photo,1.0,10.0,0.0
3,Amanda,NaN,2024-06-17,2024-06-17 11:56:39,Purple Machine,Photo,1.0,10.0,10.0
4,Amanda,NaN,2024-06-17,2024-06-17 12:28:02,Purple Machine,Photo,1.0,10.0,10.0
...,...,...,...,...,...,...,...,...,...
2505,Nicole,Jess,2024-08-22,2024-08-22 17:12:08,Purple Machine,Copy/Print,2.0,5.0,10.0
2506,Nicole,Jess,2024-08-22,2024-08-22 17:16:03,Blue Machine,Photo,1.0,10.0,10.0
2507,Nicole,Jess,2024-08-22,2024-08-22 17:21:34,Blue Machine,Copy/Print,2.0,5.0,10.0
2508,Nicole,Jess,2024-08-22,2024-08-22 17:26:11,Blue Machine,Photo,1.0,10.0,0.0


In [3]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 15 trillion tokens model 2x faster!
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # We also uploaded 4bit for 405b!
    "unsloth/Mistral-Nemo-Base-2407-bnb-4bit", # New Mistral 12b 2x faster!
    "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    "unsloth/mistral-7b-v0.3-bnb-4bit",        # Mistral v3 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
==((====))==  Unsloth 2024.9.post4: Fast Llama patching. Transformers = 4.44.2.
   \\   /|    GPU: Tesla T4. Max memory: 14.748 GB. Platform = Linux.
O^O/ \_/ \    Pytorch: 2.4.1+cu121. CUDA = 7.5. CUDA Toolkit = 12.1.
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.28.post1. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/345 [00:00<?, ?B/s]

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2024.9.post4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [5]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }
pass

from datasets import load_dataset, Dataset
dataset = Dataset.from_dict({
    'instruction': [entry['instruction'] for entry in data],
    'input': [''] * len(data),
    'output': [entry['output'] for entry in data]
}, split="train")
dataset = dataset.map(formatting_prompts_func, batched = True,)

Map:   0%|          | 0/68 [00:00<?, ? examples/s]

In [6]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 20,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

Map (num_proc=2):   0%|          | 0/68 [00:00<?, ? examples/s]

max_steps is given, it will override any value given in num_train_epochs


In [7]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 68 | Num Epochs = 3
O^O/ \_/ \    Batch size per device = 2 | Gradient Accumulation steps = 4
\        /    Total batch size = 8 | Total steps = 20
 "-____-"     Number of trainable parameters = 41,943,040


Step,Training Loss
1,1.418100
2,1.405500
3,1.423400
4,1.292100
5,1.248200
6,1.056600
7,1.046300
8,0.872200
9,0.672600
10,0.554700


In [8]:
with open('Joined_DF_0.csv', mode='r', newline='', encoding='utf-8') as csvfile:
    csvreader = csv.reader(csvfile, quotechar='"', delimiter=',')
    headers = next(csvreader)
    combined_data = ['"' + '","'.join(headers) + '"']

    for row in csvreader:
        combined_row = '"' + '","'.join(row) + '"'  # Add quotes around each cell
        combined_data.append(combined_row)

# Join all rows into a single string separated by newlines
training_input = '\n'.join(combined_data)
training_input

'"staff","replaced","Date","action_time","product","type","quantity","base","total"\n"Amanda","","2024-06-17","2024-06-17 10:10:30","Blue Machine","Photo","1.0","10.0","10.0"\n"Amanda","","2024-06-17","2024-06-17 10:13:38","Blue Machine","Photo","1.0","10.0","10.0"\n"Amanda","","2024-06-17","2024-06-17 10:24:14","Purple Machine","Photo","1.0","10.0","0.0"\n"Amanda","","2024-06-17","2024-06-17 11:56:39","Purple Machine","Photo","1.0","10.0","10.0"\n"Amanda","","2024-06-17","2024-06-17 12:28:02","Purple Machine","Photo","1.0","10.0","10.0"\n"Amanda","","2024-06-17","2024-06-17 12:49:20","Purple Machine","Photo","1.0","10.0","10.0"\n"Amanda","","2024-06-17","2024-06-17 12:52:49","Blue Machine","Photo","1.0","10.0","0.0"\n"Amanda","","2024-06-17","2024-06-17 13:03:23","Purple Machine","Photo","1.0","10.0","0.0"\n"Amanda","","2024-06-17","2024-06-17 14:23:21","Purple Machine","Photo","1.0","10.0","10.0"\n"Amanda","","2024-06-17","2024-06-17 14:27:40","Purple Machine","Copy/Print","1.0","5.0

In [9]:
test_prompts = ["give me a table of total sales grouped by staff - order it by sales",
                "how many photos were sold in august",
                "Can you produce a graph on who worked the most shifts?",
                "How many BTS albums sold in July?"
                "How many copies were sold over time?"]

In [18]:
#inference
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
model.to('cuda')
from transformers import TextStreamer
all_outputs = []

torch.cuda.empty_cache()

for prompt in test_prompts:

  inputs = tokenizer(
  [
      alpaca_prompt.format(
          prompt, # instruction
          training_input, #file content
          "", # output - leave this blank for generation!
      )
  ], return_tensors = "pt").to("cuda")
  output = model.generate(**inputs, max_new_tokens = 512)
  output_text = tokenizer.decode(output[0], skip_special_tokens=True)
  all_outputs.append(output_text)

OutOfMemoryError: CUDA out of memory. Tried to allocate 816.00 MiB. GPU 0 has a total capacity of 14.75 GiB of which 475.06 MiB is free. Process 4244 has 14.28 GiB memory in use. Of the allocated memory 14.10 GiB is allocated by PyTorch, and 28.58 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
import re
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('all-MiniLM-L6-v2')
# check multiple models




def scoring_syntax(output):
  try:
    exec(output)
    print("Success")
    return 1
  except Exception as error:
    print("Fail")
    print(f"Failed exception: {error}")
    return 0


def scoring_title(query,output):
  title = re.findall(r'\.set_title\((["\'])(.*?)\1\)', output)
  if len(title)==0:
    print('No Title found!')
    return 0
  query_embedding = model.encode(query, convert_to_tensor=True)
  title_embedding = model.encode(title, convert_to_tensor=True)

  # Compute cosine similarity
  similarity = util.cos_sim(query_embedding, title_embedding)
  return similarity.item()

syntax_score = 0
title_score = 0
for output_text in all_outputs:
  trimmed = re.search(r'### Response\s*(.*)', output_text, re.DOTALL)
  final = trimmed.group(1).strip()
  query = re.search(r'### Instruction:\s*(.*?)\s*### Input:', output_text, re.DOTALL).group(1).strip()
  s_score = scoring_syntax(final)
  t_score = scoring_title(query,final)
  # print(f'Score on the Syntax: {s_score}')
  # print(f'Score on the Title: {t_score}')
  syntax_score += s_score
  title_score += t_score

print(f"Final Score on the Syntax: {syntax_score}")
print(f"Final Score on the Title: {title_score}")






modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.78k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/686 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Fail
Failed exception: invalid syntax (<string>, line 1)
Score on the Syntax: 0
No Title found!
Score on the Title: 0


In [ ]:
#saving
model.save_pretrained("llama") # Local saving
tokenizer.save_pretrained("llama")

In [ ]:
#load:
if True:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "lora_model", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model) # Enable native 2x faster inference

# alpaca_prompt = You MUST copy from above!

inputs = tokenizer(
[
    alpaca_prompt.format(
        "What is a famous tall tower in Paris?", # instruction
        "", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128)